<a href="https://colab.research.google.com/github/astroelaa/-weather-forecasting-/blob/main/mini_rag_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Import**

In [ ]:
!pip install -q langchain-text-splitters sentence-transformers transformers chromadb pypdf python-docx

In [ ]:
import os
import glob
import shutil
import chromadb
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
for filename in uploaded.keys():
    shutil.move(filename, os.path.join(DATA_DIR, filename))

os.listdir(DATA_DIR)

Reader for the pdf

In [ ]:
from pypdf import PdfReader

reader = PdfReader("data/Savov_Notes.pdf")

text = ""
for page in reader.pages:
    text += page.extract_text() or ""

print(len(text))
print(text[:300])

chuncking

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_text(text)

len(chunks)

embed the chuncks

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
chunk_embeddings = model.encode(chunks, show_progress_bar=True)

chunk_embeddings.shape

store in Chroma

In [ ]:
client = chromadb.EphemeralClient()

try:
    client.delete_collection("notes")
except Exception:
    pass

collection = client.create_collection("notes")

collection.add(
    ids=[str(i) for i in range(len(chunks))],
    embeddings=chunk_embeddings.tolist(),
    documents=chunks,
)

collection.count()

retrieval

In [ ]:
def retrieve(query, k=3):
    query_vector = model.encode(query).tolist()
    results = collection.query(query_embeddings=[query_vector], n_results=k)

    retrieved = []
    for text_chunk, distance in zip(results["documents"][0], results["distances"][0]):
        retrieved.append({"text": text_chunk, "distance": distance})
    return retrieved

test = retrieve("ask something you know is in your notes", k=3)
for r in test:
    print(round(r["distance"], 3))
    print(r["text"][:200])
    print()

context & prompt

In [ ]:
PROMPT_TEMPLATE = """Use only the context below to answer. Do not use any outside knowledge.
If the context does not contain the answer, respond exactly: "I don't have enough information in the provided documents."

Context:
{context}

Question:
{question}

Answer:"""

def build_prompt(question, context):
    return PROMPT_TEMPLATE.format(context=context, question=question)

generation model

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

gen_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
gen_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

def generate_answer(question, context):
    prompt = build_prompt(question, context)
    inputs = gen_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    output = gen_model.generate(**inputs, max_new_tokens=200)
    return gen_tokenizer.decode(output[0], skip_special_tokens=True)

pipline to question

In [ ]:
def rag_pipeline(question, k=3):
    retrieved = retrieve(question, k=k)
    context = build_context(retrieved)
    answer = generate_answer(question, context)
    return answer

test no. 1

In [ ]:
questions = [
    "what is a diagonalizable matrix",
    "what does it mean for a matrix to be invertible",
    "what is the rank of a matrix",
]

for q in questions:
    print(q)
    print(rag_pipeline(q))
    print()

In [ ]:
off_topic_questions = [
    "what's the weather in Cairo today",
    "who won the world cup in 2018",
]

for q in off_topic_questions:
    print(q)
    print(rag_pipeline(q))
    print()

fixing formates

In [ ]:
import re

from pypdf import PdfReader

reader = PdfReader("data/Savov_Notes.pdf")

text = ""
for page in reader.pages:
    text += page.extract_text() or ""

text = re.sub(r'[˛⃗]', '', text)
text = re.sub(r'\s+', ' ', text)

print(len(text))
print(text[:300])

test no.2

In [ ]:

print()

questions_2 = [
    "what is a diagonalizable matrix",
    "what does it mean for a matrix to be invertible",
    "what is the rank of a matrix",
    "what is an eigenvector",
]

for q in questions_2:
    print(q)
    print(rag_pipeline(q))
    print()

print("Test No. 2 — off-topic questions")
print()

off_topic_2 = [
    "what's the weather in Cairo today",
    "who won the world cup in 2018",
]

for q in off_topic_2:
    print(q)
    print(rag_pipeline(q))
    print()